# jjh

---

**Project**
- CCRM

**Module**
- notebooks

**Author**
- Hyeok

**Created**
- 2026-03-07

**Purpose**
- TODO: EasyEnsemble
---


In [1]:
# [배포 전제 조건] 불균형 데이터 처리를 위한 라이브러리 설치 필요
# 실무 환경(CI/CD)에서는 requirements.txt에 명시하는 것을 권장합니다.
%pip install imbalanced-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys

from imblearn.ensemble import EasyEnsembleClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score, accuracy_score
)
from sklearn.model_selection import train_test_split

In [3]:
# 1. 환경 설정 및 데이터 로드
# 사내 공통 모듈(common) 참조를 위해 시스템 경로 설정
sys.path.append(os.path.abspath("../../"))
from common.fetch_creditcard import fetch_creditcard

ModuleNotFoundError: No module named 'common'

In [ ]:
# [배포 시 고려사항] 데이터 분할
# stratify=y: 타겟 불균형이 심한 '이탈 데이터' 특성상, Train/Test셋 내 정답 비율을 유지하기 위함
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 2. [📊 데이터 프로파일링] 팀 공통 규격 출력
# 모델 학습 전, 실제 분포가 의도대로 나누어졌는지 검증하여 '데이터 왜곡'으로 인한 편향 문제 예방
print(f"\n{'Train / Test Split Summary':^50}")
print("="*50)
print(f"X_train shape : {X_train.shape}")
print(f"X_test  shape : {X_test.shape}")
print("-"*50)
print("Target Distribution (Train)")
print(y_train.value_counts().to_string())
print("-"*50)
print("Train Ratio")  # 정답 비율이 stratify를 통해 동일하게 배분되었는지 확인
print(y_train.value_counts(normalize=True).to_string())
print("="*50)

In [ ]:
# 3. 모델 생성 및 학습: EasyEnsembleClassifier
# 불균형 데이터에 특화된 Bagging 방식 적용 (앙상블 내 각 모델이 다운샘플링된 데이터 학습)
# n_estimators=10: 과적합 방지와 학습 시간 사이의 균형을 고려한 설정
# n_jobs=-1: 가용 CPU 코어를 모두 사용하여 대규모 데이터셋 학습 속도 최적화
model = EasyEnsembleClassifier(
    n_estimators=10, 
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

In [ ]:
# 4. 예측 및 임계값(Threshold) 조정
# 이탈 예측은 단순 정확도보다 '이탈할 사람을 놓치지 않는 것(Recall)'이 중요하므로 임계값 튜닝 수행
y_prob = model.predict_proba(X_test)[:, 1]

# threshold=0.45: 기본값(0.5)보다 낮게 설정하여 이탈 징후가 있는 고객을 더 공격적으로 포착하도록 유도
threshold = 0.45
y_pred = (y_prob >= threshold).astype(int)

In [ ]:
# 5. 결과 출력 및 모니터링
# PR-AUC: 데이터 불균형이 심할 때 Accuracy보다 모델의 실무적 성능을 더 정확히 평가하는 지표
print("\n### BankChurner 모델 결과 ###")
print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}")
print(f"PR-AUC   : {average_precision_score(y_test, y_prob):.4f}")

# Classification Report: Precision, Recall, F1-Score를 통해 군집별 성능 불균형 여부 최종 점검
print("\n[Classification Report]")
print(classification_report(y_test, y_pred))
print("="*50)

# [유지보수 팁] 향후 데이터가 업데이트될 경우, 데이터 소스의 결측치(NaN) 여부에 따라
# X.fillna(0) 또는 SimpleImputer를 활용한 전처리 로직이 모델 상단에 추가되어야 안전합니다.